In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nikhil8885/nonconv-hindi-medical-20m/nonconv_hindi_medical_20M_hindi_only.jsonl


In [2]:
!pip install -q transformers peft trl accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 9.9 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login

login(token="hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

In [4]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    TrainerCallback   # 👈 ADD THIS
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [5]:
class PrintProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            step = state.global_step
            epoch = round(state.epoch, 2) if state.epoch else 0
            
            msg = f"Step {step}"
            if epoch:
                msg += f" | Epoch {epoch}"
            
            if "loss" in logs:
                msg += f" | Train Loss: {logs['loss']:.4f}"
            if "eval_loss" in logs:
                msg += f" | Eval Loss: {logs['eval_loss']:.4f}"
            
            print(msg)

In [6]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print("Memory:", round(gpu.total_memory / 1024**3, 2), "GB")

GPU: Tesla T4
Memory: 14.56 GB


In [7]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [10]:
dataset = load_dataset("json", data_files="/kaggle/input/datasets/nikhil8885/nonconv-hindi-medical-20m/nonconv_hindi_medical_20M_hindi_only.jsonl")

dataset = dataset["train"].train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

Generating train split: 0 examples [00:00, ? examples/s]

In [11]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/results",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,

    logging_steps=10,
    logging_strategy="steps",

   
    eval_steps=50,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    fp16=True,
    optim="adamw_torch",
    report_to="none"
)

In [12]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    callbacks=[PrintProgressCallback()],  # 👈 ADD THIS
)

Adding EOS to train dataset:   0%|          | 0/7510 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7510 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2294 > 2048). Running this sequence through the model will result in indexing errors


Adding EOS to eval dataset:   0%|          | 0/835 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/835 [00:00<?, ? examples/s]

In [13]:
import os

checkpoint_dir = "/kaggle/working/results"

if os.path.exists(checkpoint_dir) and len(os.listdir(checkpoint_dir)) > 0:
    print("Resuming from checkpoint...\n")
    trainer.train(resume_from_checkpoint=True)
else:
    print("Starting fresh training...\n")
    trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting fresh training...



Step,Training Loss
10,1.159393
20,1.162453
30,1.093795
40,1.109966
50,1.088479
60,1.082519
70,1.120690
80,1.092810
90,1.055021
100,1.073154


Step 10 | Epoch 0.01 | Train Loss: 1.1594
Step 20 | Epoch 0.01 | Train Loss: 1.1625
Step 30 | Epoch 0.02 | Train Loss: 1.0938
Step 40 | Epoch 0.02 | Train Loss: 1.1100
Step 50 | Epoch 0.03 | Train Loss: 1.0885
Step 60 | Epoch 0.03 | Train Loss: 1.0825
Step 70 | Epoch 0.04 | Train Loss: 1.1207
Step 80 | Epoch 0.04 | Train Loss: 1.0928
Step 90 | Epoch 0.05 | Train Loss: 1.0550
Step 100 | Epoch 0.05 | Train Loss: 1.0732
Step 110 | Epoch 0.06 | Train Loss: 1.0867
Step 120 | Epoch 0.06 | Train Loss: 1.0683
Step 130 | Epoch 0.07 | Train Loss: 1.1143
Step 140 | Epoch 0.07 | Train Loss: 1.0834
Step 150 | Epoch 0.08 | Train Loss: 1.1173
Step 160 | Epoch 0.09 | Train Loss: 1.0475
Step 170 | Epoch 0.09 | Train Loss: 1.0785
Step 180 | Epoch 0.1 | Train Loss: 1.0782
Step 190 | Epoch 0.1 | Train Loss: 1.0624
Step 200 | Epoch 0.11 | Train Loss: 1.0551
Step 210 | Epoch 0.11 | Train Loss: 1.0875
Step 220 | Epoch 0.12 | Train Loss: 1.0857
Step 230 | Epoch 0.12 | Train Loss: 1.0750
Step 240 | Epoch 0.13 

In [14]:
trainer.save_model("/kaggle/working/final_model")